## Integrador API OPTA

In [1]:
import requests
import re
from bs4 import BeautifulSoup
import os
import json
from datetime import datetime
import time
import random
import pandas as pd
import re
from datetime import datetime, timedelta,date
import warnings  # Importa el módulo warnings, que maneja los mensajes de advertencia en Python.
import sw_scraping_fun as swf
import sw_json2csv_fun as swj


with open("config/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

ruta_dest=config['dummy_ori']
liga_seleccionada= config['dummy_competition']

warnings.filterwarnings("ignore")



In [2]:
def extraer_datos_partido(match,fixtures,token):
    df_game=swf.get_datos_partido(match,fixtures,token)
    data=swj.get_events(df_game)

    return data

# Scraping + Parseo

In [3]:
metadata = pd.read_excel("config/metadata.xlsx")
metadata=metadata[metadata.competicion_id==liga_seleccionada]

## API -> JSON

Se extrae un fichero json de cada partido. Posteriormente, se parsea y se transforma en csv.
Un partido contiene la siguiente información, a nivel partido:
- *matchdata*: atributos del partido
- *playerdata*: atributos de jugadores
- *teamdata*: atributos de los equipos
- *eventdata*: eventing

Adicionalmente, se incorporan datos agregados de equipos (teamstats), extraidos mediante selenium.

In [4]:
ficheros = [f for f in os.listdir(os.path.join(os.getcwd(),"fixtures")) if f.endswith(('.xlsx', '.xls')) and "$" not in f]
fixtures = pd.DataFrame()
for i in ficheros:
    fi = pd.read_excel(os.path.join(os.getcwd(),"fixtures",i))
    fixtures=pd.concat([fixtures,fi])

In [5]:
all_fixtures_df=pd.DataFrame()
for i,j in metadata.iterrows():
        ruta_dest = j['origen']
        season= j['season']
        comp = j['competicion_id']
        #comps.append(comp)
        #seasons.append(season)
        fixtures_url = j['url']
        fichero_liga = "matches_{}_{}.xlsx".format(comp.lower(),season)
    
    
        
        if fichero_liga in ficheros:
            all_fixtures=pd.read_excel(os.path.join(os.getcwd(),"fixtures")+"/{}".format(fichero_liga))
            all_fixtures = all_fixtures.drop_duplicates(subset="match_id",keep='first')
        else:
            all_fixtures = swf.scrape_fixtures(metadata,fixtures_url)
            if all_fixtures.shape[0]>0:
                all_fixtures['season'] = season
                all_fixtures['competition'] = comp
        all_fixtures['date'] = all_fixtures['date'].fillna(all_fixtures[all_fixtures['date'].isna()==False]['date'].values[-1])
        all_fixtures['dia_id']=all_fixtures.date.apply(lambda x: datetime.fromisoformat(x.replace("Z", "")).date())
        all_fixtures_df=pd.concat([all_fixtures_df,all_fixtures])

torneo_name: esp-laligaeasports_2025-2026
torneo_id: 80zg2v1cuqcfhphn56u4qpyqc
✅ Estructura de carpetas creada (si no existía).
Error: Añadiendo token por defecto
Leyendo sdapi_outlet_key desde config/config.json...
sdapi_outlet_key: ft1tiv1inq7v1sk3y9tv12yh5
Fixture guardado exitosamente.
No existen datos extra: fixtures
No existen datos extra: results


In [16]:
df_partidos = all_fixtures_df[all_fixtures_df.dia_id<date.today()]

print("Partidos ya jugados: {} ({} totales en {})".format(df_partidos[df_partidos.competition==liga_seleccionada].shape[0],all_fixtures_df[all_fixtures_df.competition==liga_seleccionada].shape[0],liga_seleccionada))

Partidos ya jugados: 168 (240 totales en esp-ligaf)


In [7]:
df_partidos.head()

,competition,season,date,home_team,away_team,stadium,match_id,URL,dia_id
72,esp-ligaf,2025-2026,2026-02-22Z,Sevilla,Atlético Madrid,NaN,8bo8kwuy2uvdmay9yf6dq9ses,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-22
73,esp-ligaf,2025-2026,2026-02-22Z,Real Sociedad,Espanyol,NaN,8cuardpf2jimws64c0qsc0xlg,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-22
74,esp-ligaf,2025-2026,2026-02-22Z,Eibar,Athletic Club,NaN,8b8qojrl7pmleq86uh9y0bhn8,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-22
75,esp-ligaf,2025-2026,2026-02-22Z,Real Madrid,Tenerife,NaN,8agzaoxboug1td3uvjrg6obh0,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-22
76,esp-ligaf,2025-2026,2026-02-21Z,Granada,Barcelona,NaN,8d879ol25noswi79mfshf9rf8,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-21


In [8]:
partido = df_partidos.match_id.values[-1]
partido

'1f9e25njwyjk6ruheatqgglw'

In [9]:
data=extraer_datos_partido(
                    partido,
                    df_partidos,
                    swf.obtener_sdapi_outlet_key(metadata.url.values[0]))

Error: Añadiendo token por defecto
Leyendo sdapi_outlet_key desde config/config.json...
https://api.performfeeds.com/soccerdata/matchevent/ft1tiv1inq7v1sk3y9tv12yh5/1f9e25njwyjk6ruheatqgglw?_rt=c&_lcl=en&_fmt=jsonp&sps=widgets&_clbk=W3e14cbc3e4b2577e854bf210e5a3c7028c7409678
⏳ Esperando 5.84 segundos antes de descargar (intento 1/3)...
✅ Guardado: 2025-08-15Z_Girona_Rayo Vallecano_1f9e25njwyjk6ruheatqgglw.json


In [10]:
data.keys()

dict_keys(['eventData', 'matchData', 'teamData', 'playerData'])

#### Eventos

In [11]:
events= data['eventData']
print("Nº filas: {}".format(events.shape[0]))
events.head()

Nº filas: 1716


,id,eventId,typeId,periodId,timeMin,timeSec,contestantId,outcome,x,y,...,relatedEventId,relatedPlayerId,is_penalty,isGoal,minute,second,oppositionTeamName,time_seconds,position,isFirstEleven
0,2835039373,1,34,16,0,0,3budh3j9xivsid3ptm8ptpy4k,1,0.0,0.0,...,NaN,NaN,NaN,0,0,0,Girona,0,NaN,NaN
1,2835040341,1,34,16,0,0,7h7eg7q7dbwvzww78h9d5eh0h,1,0.0,0.0,...,NaN,NaN,NaN,0,0,0,Rayo Vallecano,0,NaN,NaN
2,2835057147,2,32,1,0,0,7h7eg7q7dbwvzww78h9d5eh0h,1,0.0,0.0,...,NaN,NaN,NaN,0,0,0,Rayo Vallecano,0,NaN,NaN
3,2835057151,2,32,1,0,0,3budh3j9xivsid3ptm8ptpy4k,1,0.0,0.0,...,NaN,NaN,NaN,0,0,0,Girona,0,NaN,NaN
4,2835057157,3,1,1,0,0,3budh3j9xivsid3ptm8ptpy4k,1,49.8,49.9,...,NaN,NaN,NaN,0,0,0,Girona,0,FW,1.0


#### Dimensiones:

- Partido
- Equipos
- Jugadores

In [12]:
df_match= data['matchData']
df_match.head()

,matchId,id,coverageLevel,date,time,localDate,localTime,week,numberOfPeriods,periodLength,...,away_position,competition_id,competition_name,competition_knownName,competition_sponsorName,competition_competitionCode,competition_competitionFormat,venueName,expandedMaxMinute,matchStatus
0,1f9e25njwyjk6ruheatqgglw,1f9e25njwyjk6ruheatqgglw,13,2025-08-15Z,17:00:00Z,2025-08-15,19:00:00,1,2,45,...,away,34pl8szyvrbwcmfkuocjm3r6t,Primera División,Spanish La Liga,LALIGA EA SPORTS,PRD,Domestic league,Estadi Municipal de Montilivi,104,Played


In [13]:
df_teams= data['teamData']
df_teams.head()

,teamId,teamName,shortName,officialName,code,field,country,matchId,scores.halftime,scores.fulltime
0,7h7eg7q7dbwvzww78h9d5eh0h,Girona,Girona,Girona FC,GIR,home,Spain,1f9e25njwyjk6ruheatqgglw,0,1
1,3budh3j9xivsid3ptm8ptpy4k,Rayo Vallecano,Rayo,Rayo Vallecano de Madrid,RAY,away,Spain,1f9e25njwyjk6ruheatqgglw,3,3


In [14]:
df_players= data['playerData']
print("Nº filas: {}".format(df_players.shape[0]))
df_players.head()

Nº filas: 45


,field_position,playerId,value_TeamPlayerFormation,shirtNo,value_TeamFormation,teamId,matchId,field,isFirstEleven,position,playerName,subbedOutPlayerId,subbedInExpandedMinute,subbedInPeriod_value,subbedInPlayerId,subbedOutExpandedMinute,subbedOutPeriod_value
0,1,9h6xf4ac4rpsjpummcdx26z9,1,13,8,3budh3j9xivsid3ptm8ptpy4k,1f9e25njwyjk6ruheatqgglw,away,1,GK,A. Batalla,NaN,NaN,NaN,NaN,NaN,NaN
1,2,38dhmhce7deuqgqmc3p7nhai2,2,2,8,3budh3j9xivsid3ptm8ptpy4k,1f9e25njwyjk6ruheatqgglw,away,1,DR,A. Rațiu,NaN,NaN,NaN,NaN,NaN,NaN
2,2,amt4voyjzhnvr55t08hzlw50q,3,3,8,3budh3j9xivsid3ptm8ptpy4k,1f9e25njwyjk6ruheatqgglw,away,1,DC,Pep Chavarría,NaN,NaN,NaN,NaN,NaN,NaN
3,3,u04ao1ldrjjesgqq52o7ghqt,4,17,8,3budh3j9xivsid3ptm8ptpy4k,1f9e25njwyjk6ruheatqgglw,away,1,MC,Unai López,NaN,NaN,NaN,8onruhr4es313aq52xsl7nxlh,80.0,2.0
4,2,533095rnz0k8mxqe6gzn0mfo5,5,24,8,3budh3j9xivsid3ptm8ptpy4k,1f9e25njwyjk6ruheatqgglw,away,1,DL,F. Lejeune,NaN,NaN,NaN,NaN,NaN,NaN


#### Datos Agregados por Equipo

In [15]:
df_teamstats=swf.get_teamstats(df_partidos[df_partidos.match_id==df_partidos.match_id.values[-1]],ruta_dest)
print("Nº filas: {}".format(df_teamstats.shape[0]))
df_teamstats.head()

Error Message: Unable to obtain chromedriver using Selenium Manager; Message: Unsuccessful command executed: C:\Users\aleja\AppData\Roaming\Python\Python310\site-packages\selenium\webdriver\common\windows\selenium-manager.exe --browser chrome --output json.
The chromedriver version cannot be discovered
; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location



UnboundLocalError: local variable 'df_pivot' referenced before assignment